# SSI FastConnect — kiểm tra OHLCV cho TradingAgents

Notebook này thay thế đoạn tự gọi REST endpoint đang trả `HTTP 400` bằng **Python SDK chính thức `ssi-sdk`**. Chỉ đọc dữ liệu thị trường của mã FPT; không có lệnh mua/bán.

### Cách dùng
1. Mở notebook bằng Google Colab và chọn **Runtime → Run all**.
2. Nếu chưa lưu credentials trong Colab Secrets, notebook sẽ yêu cầu nhập Client ID, API Key và API Secret theo dạng ẩn. Có thể lưu sẵn trong **Secrets** với tên `SSI_CLIENT_ID`, `SSI_API_KEY`, `SSI_API_SECRET` để khỏi nhập lại ở các phiên sau.
3. Không dán hoặc in token/secret vào chat, log hay GitHub.

Nguồn API: [SSI Python SDK](https://developers.ssi.com.vn/docs/sdk/python/service-classes), [hướng dẫn OHLC](https://developers.ssi.com.vn/docs/tutorials/ohlc).

In [ ]:
%pip -q install "ssi-sdk==3.3.2" pandas

In [ ]:
import os
import re
from getpass import getpass

def _colab_secret(name):
    """Read a Colab Secret if available; never print its value."""
    value = os.environ.get(name)
    if value:
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None

def _credential(name, prompt):
    value = _colab_secret(name)
    if value:
        print(f"{name}: đã nạp từ môi trường/Colab Secrets (giá trị được ẩn).")
        return value
    print(f"Chưa tìm thấy {name} trong Colab Secrets; nhập một lần cho phiên này.")
    return getpass(prompt).strip()

CLIENT_ID = _credential("SSI_CLIENT_ID", "SSI Client ID (không hiển thị): ")
API_KEY = _credential("SSI_API_KEY", "SSI API Key (không hiển thị): ")
API_SECRET = _credential("SSI_API_SECRET", "SSI API Secret (không hiển thị): ")

if not all((CLIENT_ID, API_KEY, API_SECRET)):
    raise ValueError("Thiếu Client ID/API Key/API Secret. Hãy kiểm tra lại Colab Secrets hoặc nhập lại.")

print("Đã nhận đủ thông tin xác thực. Không có credential nào được in ra.")

In [ ]:
from datetime import date, timedelta
import pandas as pd
from ssi_sdk import Auth, Data, Config

SYMBOL = "FPT"
TO_DATE = date.today().strftime("%Y/%m/%d") + " 23:59:59"
FROM_DATE = (date.today() - timedelta(days=60)).strftime("%Y/%m/%d") + " 00:00:00"

config = Config(
    client_id=CLIENT_ID,
    api_key=API_KEY,
    api_secret=API_SECRET,
    # Không cần private key cho Market Data; notebook không gọi Trading API.
)

def _safe_exception_message(exc):
    """Keep useful error context while removing credentials/tokens if present."""
    message = str(exc)
    for secret in (CLIENT_ID, API_KEY, API_SECRET):
        if secret:
            message = message.replace(secret, "[REDACTED]")
    message = re.sub(
        r"(?i)(authorization\s*[:=]\s*bearer\s+)[^\s,;]+",
        r"\1[REDACTED]",
        message,
    )
    message = re.sub(
        r"eyJ[A-Za-z0-9_-]{15,}\.[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]+",
        "[REDACTED_TOKEN]",
        message,
    )
    return message[:800]

print(f"Đang đọc OHLC ngày cho {SYMBOL}, từ {FROM_DATE} đến {TO_DATE}...")
try:
    with Auth(config) as auth:
        auth.authenticate()  # Market Data không yêu cầu OTP.
        with Data(auth) as data:
            bars = data.market_data.get_ohlc_1day_historical(
                symbol=SYMBOL,
                from_date=FROM_DATE,
                to_date=TO_DATE,
                page=1,
                size=100,
            )
except Exception as exc:
    print(f"SSI Market Data thất bại — loại lỗi: {type(exc).__name__}")
    print(f"Chi tiết đã lọc thông tin nhạy cảm: {_safe_exception_message(exc)}")
    raise RuntimeError("Không lấy được OHLCV qua SSI SDK; xem loại lỗi và chi tiết đã lọc ở trên.") from None

print(f"Xác thực và gọi SSI SDK hoàn tất. Số nến trả về: {len(bars or [])}")

In [ ]:
def _field(obj, name):
    return obj.get(name) if isinstance(obj, dict) else getattr(obj, name, None)

columns = ["time", "symbol", "open", "high", "low", "close", "volume", "value"]
records = []
for bar in (bars or []):
    records.append({
        "time": _field(bar, "trading_date"),
        "symbol": _field(bar, "symbol") or SYMBOL,
        "open": _field(bar, "open_price"),
        "high": _field(bar, "high_price"),
        "low": _field(bar, "low_price"),
        "close": _field(bar, "close_price"),
        "volume": _field(bar, "volume"),
        "value": _field(bar, "value"),
    })

ohlcv_df = pd.DataFrame(records, columns=columns)
if ohlcv_df.empty:
    print("SSI không trả nến trong khoảng ngày đã chọn. Xác thực/API call không báo lỗi; thử kiểm tra lại mã hoặc khoảng ngày trong bước tiếp theo.")
else:
    ohlcv_df["time"] = pd.to_datetime(ohlcv_df["time"], errors="coerce")
    ohlcv_df = ohlcv_df.sort_values("time").reset_index(drop=True)
    display(ohlcv_df.tail(20))
    print(f"✅ Đọc OHLCV thành công: {len(ohlcv_df)} nến; cột đã chuẩn hóa cho bước tích hợp TradingAgents.")